In [1]:
!pip install gdown
!gdown --id https://drive.google.com/file/d/1eRHVSgTL4c-5UB_xva9XkGJ8cLZM67A4/view?usp=sharing
!gdown --id 1DKRzcff89Naeg1KWwjZy80Z2JOTPmMNf

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=https://drive.google.com/file/d/1eRHVSgTL4c-5UB_xva9XkGJ8cLZM67A4/view?usp=sharing

but Gdown can't. Please check connections and permissions.
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Failed to retrieve file url:

	Too many users have viewed or downloaded this file

In [11]:
!unzip -q EmoVid_Data.zip -d ./emovid_data
!unzip -q emotion-embedder.zip -d ./emo_emb
!pip install sentence_transformers

unzip:  cannot find or open EmoVid_Data.zip, EmoVid_Data.zip.zip or EmoVid_Data.zip.ZIP.
replace ./emo_emb/config_sentence_transformers.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: Requirement already satisfied: sentence_transformers in /usr/local/lib/python3.12/dist-packages (5.5.1)


In [102]:
import pandas as pd
import numpy as np
data = pd.read_csv('emo_vid_emoted_captions.csv')
X = data['emoted_caption']
y = data['label']
X.shape, y.shape


((13281,), (13281,))

In [103]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size = 0.8, random_state = 42, stratify = y)

unique_emotions = sorted(list(set(y_train)))
emotion_to_id = {emotion: idx for idx, emotion in enumerate(unique_emotions)}
y_train = np.array([emotion_to_id[y] for y in y_train], dtype='int64')
y_test = np.array([emotion_to_id[y] for y in y_test], dtype='int64')
X_train.shape, y_train.shape, y_train[:10], y_test[:10]


((10624,),
 (10624,),
 array([4, 4, 1, 5, 1, 0, 4, 0, 7, 2]),
 array([7, 1, 3, 0, 4, 1, 6, 1, 3, 6]))

In [104]:
from sentence_transformers import SentenceTransformer

ft_bert = SentenceTransformer("/content/emo_emb", device="cuda")


print(ft_bert.encode(list(X_train[:10])).shape)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

(10, 768)


In [105]:
import torch
import torch.nn as nn

class LinearClassifier(nn.Module):
  def __init__(self):
    super(LinearClassifier, self).__init__()
    self.lin1 = nn.Linear(768, 256)
    self.ln1 = nn.LayerNorm(256)
    self.act1 = nn.ReLU()

    self.drop1 = nn.Dropout(0.3)

    self.lin2 = nn.Linear(256, 48)
    self.ln2 = nn.LayerNorm(48)
    self.act2 = nn.ReLU()

    self.drop2 = nn.Dropout(0.2)

    self.lin3 = nn.Linear(48, 8)

  def forward(self, x):
    x = self.lin1(x)
    x = self.ln1(x)
    x = self.act1(x)
    x = self.drop1(x)
    x = self.lin2(x)
    x = self.ln2(x)
    x = self.act2(x)
    x = self.drop2(x)
    x = self.lin3(x)
    return x


In [109]:
import torch.optim as opt
model = LinearClassifier().to('cuda')
optim = opt.AdamW(model.parameters(), lr = 1e-4)
custom_weights = [
    1.4,
    1.0,
    1.7,
    1.0,
    0.7,
    1.1,
    0.8,
    1.0
]
class_weights_tensor = torch.tensor(custom_weights, dtype=torch.float32).to('cuda')

loss = nn.CrossEntropyLoss(weight=class_weights_tensor)
num_epochs = 1000

X_train = ft_bert.encode(list(X_train))
for epoch in range(num_epochs):
  model.train()
  loss_arr = []
  for batch_idx in range(0, len(X_train)// 100):
    optim.zero_grad()

    X_batch = torch.from_numpy(X_train[100 * batch_idx : 100 * (batch_idx + 1)]).to('cuda')
    y_batch = torch.from_numpy(np.array(y_train[100 * batch_idx : 100 * (batch_idx + 1)])).to('cuda')

    pred = model(X_batch)
    l = loss(pred, y_batch)
    l.backward()
    optim.step()
    loss_arr.append(l.item())

  print(f"Epoch {epoch + 1}/ {num_epochs} || Loss: {sum(loss_arr) / len(loss_arr)}")



Epoch 1/ 1000 || Loss: 1.628281831741333
Epoch 2/ 1000 || Loss: 1.4139606266651514
Epoch 3/ 1000 || Loss: 1.3227276093554947
Epoch 4/ 1000 || Loss: 1.2547398783125967
Epoch 5/ 1000 || Loss: 1.1971969593246028
Epoch 6/ 1000 || Loss: 1.14886622496371
Epoch 7/ 1000 || Loss: 1.1157136137755412
Epoch 8/ 1000 || Loss: 1.0750862067600466
Epoch 9/ 1000 || Loss: 1.0500717354270648
Epoch 10/ 1000 || Loss: 1.0235882058458508
Epoch 11/ 1000 || Loss: 1.0046868717895363
Epoch 12/ 1000 || Loss: 0.9828906424765317
Epoch 13/ 1000 || Loss: 0.9637826124452195
Epoch 14/ 1000 || Loss: 0.9545174094865907
Epoch 15/ 1000 || Loss: 0.9382724992509158
Epoch 16/ 1000 || Loss: 0.9263884295832436
Epoch 17/ 1000 || Loss: 0.9115767776966095
Epoch 18/ 1000 || Loss: 0.9003597216786079
Epoch 19/ 1000 || Loss: 0.889497088373832
Epoch 20/ 1000 || Loss: 0.8774831964159912
Epoch 21/ 1000 || Loss: 0.8747235182321297
Epoch 22/ 1000 || Loss: 0.8654641694617722
Epoch 23/ 1000 || Loss: 0.8586101970582638
Epoch 24/ 1000 || Loss: 

In [111]:
from sklearn.metrics import classification_report
model.eval()
pred = model(torch.from_numpy(ft_bert.encode(np.array(X_test))).to('cuda'))
pred = torch.argmax(pred, dim=1)

print(classification_report(y_test, pred.cpu().numpy()))


              precision    recall  f1-score   support

           0       0.74      0.77      0.76       231
           1       0.80      0.87      0.83       583
           2       0.79      0.68      0.73       219
           3       0.84      0.89      0.86       271
           4       0.95      0.95      0.95       260
           5       0.87      0.82      0.84       300
           6       0.93      0.91      0.92       481
           7       0.81      0.78      0.79       312

    accuracy                           0.85      2657
   macro avg       0.84      0.83      0.84      2657
weighted avg       0.85      0.85      0.84      2657



In [113]:
torch.save(model.state_dict(), "emotion_classifier_head_best.pt")